In [1]:
"""
Treasury Yield Curve as Predictor of Next-Day SPY Realized Volatility
Layers 1-2: Statistical Association + Out-of-Sample Predictive Value

Per the Evaluation Protocol (framework/evaluation_protocol.md):
- Target: SPY Garman-Klass annualized volatility
- Horizon: t+1 (next trading day)
- Signal: 10Y-2Y Treasury spread at day t
- Thresholds: Layer 1 p < 0.05 (n >= 250);
              Layer 2 dR2 >= 0.02 vs naive, Diebold-Mariano p < 0.10
"""

import os
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg2
from dotenv import load_dotenv
from scipy import stats
import matplotlib.pyplot as plt

REPO_ROOT = Path(r"C:\Users\marcm\datascience\DS\Projects\market-research-lab")
load_dotenv(REPO_ROOT / ".env")

assert os.getenv("POSTGRES_DB"), "Could not locate .env — check REPO_ROOT path"
print(f"Repo root: {REPO_ROOT}")
print(f"Working with database: {os.getenv('POSTGRES_DB')}")

Repo root: C:\Users\marcm\datascience\DS\Projects\market-research-lab
Working with database: quant_research


In [2]:
def get_db_connection():
    return psycopg2.connect(
        host=os.getenv("POSTGRES_HOST"),
        port=os.getenv("POSTGRES_PORT"),
        dbname=os.getenv("POSTGRES_DB"),
        user=os.getenv("POSTGRES_USER"),
        password=os.getenv("POSTGRES_PASSWORD"),
    )


# Join the yield curve spread (from source_signals) with SPY GK volatility
# (from volatility_metrics) on date.
sql = """
    SELECT
        s.date,
        s.value AS spread_10y_2y,
        v.value AS spy_gk_vol
    FROM source_signals s
    JOIN volatility_metrics v
      ON s.date = v.date
     AND v.ticker = 'SPY'
     AND v.metric_name = 'garman_klass_annualized_pct'
    WHERE s.source_name = 'yield_curve'
      AND s.signal_name = 'spread_10y_2y'
    ORDER BY s.date ASC
"""

with get_db_connection() as conn:
    df = pd.read_sql(sql, conn)

print(f"Loaded {len(df)} joined rows")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

Loaded 4082 joined rows
Date range: 2010-01-04 to 2026-05-08


C:\Users\marcm\AppData\Local\Temp\ipykernel_21336\3788267565.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,date,spread_10y_2y,spy_gk_vol
0,2010-01-04,2.76,16.785425
1,2010-01-05,2.76,7.570115
2,2010-01-06,2.84,5.276086
3,2010-01-07,2.82,9.644474
4,2010-01-08,2.87,7.391669


In [3]:
# Construct the predictor-target pair
# - Predictor: 10Y-2Y spread on day t
# - Target: SPY GK volatility on day t+1
#
# Shift the target UP by one row so each row reads:
# "Today's yield spread -> tomorrow's volatility."

df = df.sort_values("date").reset_index(drop=True)
df["spy_gk_vol_next"] = df["spy_gk_vol"].shift(-1)

# Drop the final row (no t+1 target available)
df_pairs = df.dropna(subset=["spy_gk_vol_next"]).copy()

print(f"Original rows: {len(df)}")
print(f"After alignment: {len(df_pairs)}")
print()
print("First 5 rows:")
print(df_pairs[["date", "spread_10y_2y", "spy_gk_vol", "spy_gk_vol_next"]].head())
print()
print("Last 5 rows:")
print(df_pairs[["date", "spread_10y_2y", "spy_gk_vol", "spy_gk_vol_next"]].tail())

Original rows: 4082
After alignment: 4081

First 5 rows:
         date  spread_10y_2y  spy_gk_vol  spy_gk_vol_next
0  2010-01-04           2.76   16.785425         7.570115
1  2010-01-05           2.76    7.570115         5.276086
2  2010-01-06           2.84    5.276086         9.644474
3  2010-01-07           2.82    9.644474         7.391669
4  2010-01-08           2.87    7.391669         8.176222

Last 5 rows:
            date  spread_10y_2y  spy_gk_vol  spy_gk_vol_next
4076  2026-05-01           0.51    6.784919        10.773652
4077  2026-05-04           0.50   10.773652         4.785524
4078  2026-05-05           0.50    4.785524         7.031687
4079  2026-05-06           0.49    7.031687         8.583497
4080  2026-05-07           0.49    8.583497         3.954466


In [4]:
"""
Layer 1: Statistical Association Tests

Tests:
- Pearson correlation
- Spearman rank correlation
- Granger causality at lags 1, 5, 10
- Threshold: p < 0.05 with n >= 250
"""

from statsmodels.tsa.stattools import grangercausalitytests

x = df_pairs["spread_10y_2y"].values
y = df_pairs["spy_gk_vol_next"].values
n = len(df_pairs)

print(f"Sample size: {n}")
print(f"Protocol minimum (n >= 250): {'✅ met' if n >= 250 else '❌ not met'}")
print()

# Pearson correlation (linear relationship)
pearson_r, pearson_p = stats.pearsonr(x, y)
print(f"Pearson correlation:   r = {pearson_r:.4f}, p-value = {pearson_p:.4e}")

# Spearman rank correlation (monotonic, robust to outliers)
spearman_r, spearman_p = stats.spearmanr(x, y)
print(f"Spearman correlation:  ρ = {spearman_r:.4f}, p-value = {spearman_p:.4e}")

print()
print("Correlation interpretation:")
print(f"  Pearson:  {'✅ significant (p < 0.05)' if pearson_p < 0.05 else '❌ not significant'}")
print(f"  Spearman: {'✅ significant (p < 0.05)' if spearman_p < 0.05 else '❌ not significant'}")

# Granger causality
print()
print("Granger causality tests:")
data_for_granger = df_pairs[["spy_gk_vol_next", "spread_10y_2y"]].values

for lag in [1, 5, 10]:
    result = grangercausalitytests(data_for_granger, maxlag=[lag], verbose=False)
    f_stat = result[lag][0]["ssr_ftest"][0]
    f_pval = result[lag][0]["ssr_ftest"][1]
    verdict = "✅ significant" if f_pval < 0.05 else "❌ not significant"
    print(f"  Lag {lag:2d}: F = {f_stat:8.2f}, p = {f_pval:.4e}  {verdict}")

Sample size: 4081
Protocol minimum (n >= 250): ✅ met

Pearson correlation:   r = -0.0413, p-value = 8.2468e-03
Spearman correlation:  ρ = -0.0426, p-value = 6.4520e-03

Correlation interpretation:
  Pearson:  ✅ significant (p < 0.05)
  Spearman: ✅ significant (p < 0.05)

Granger causality tests:
  Lag  1: F =     1.29, p = 2.5554e-01  ❌ not significant
  Lag  5: F =     2.71, p = 1.9041e-02  ✅ significant
  Lag 10: F =     2.28, p = 1.1866e-02  ✅ significant


c:\Users\marcm\datascience\DS\Projects\market-research-lab\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
